# Inspect native catalogs and sky reconstruction

Complete the search described in this directory first. This notebook reads the current Parquet catalog and uses explicit input/output angular units.


In [ ]:
from pathlib import Path
import os
import sys
import numpy as np
import matplotlib.pyplot as plt

# Launch from this source checkout (or its examples directory).
candidates = [Path.cwd(), Path.cwd() / "pycwb", *Path.cwd().parents]
repo = next((p for p in candidates if (p / "examples/demo/user_parameters.yaml").is_file()), None)
if repo is None:
    raise RuntimeError("Open this notebook from the matching PycWB source checkout.")
sys.path.insert(0, str(repo / "examples"))
from notebook_support import prepare_injection_config
from pycwb.modules.logger import logger_init
logger_init()
%matplotlib inline


In [ ]:
import pandas as pd
from pycwb.modules.catalog import Catalog

# First complete a search in this example directory, or set PYCWB_EXAMPLE_RUN.
run = Path(os.environ.get("PYCWB_EXAMPLE_RUN", ".")).resolve()
cat = Catalog.open(str(run / "catalog/catalog.parquet"))
events = cat.triggers().to_pandas()
print("Triggers:", len(events), "Jobs:", len(cat.jobs))
print(events[["id", "rho", "net_cc", "ra", "dec"]].head())


In [ ]:
from astropy.coordinates import SkyCoord
import astropy.units as u

rows = cat.query("""SELECT id, rho, ra, dec, injection.ra AS inj_ra,
    injection.dec AS inj_dec FROM triggers WHERE injection IS NOT NULL""").to_pandas()
rows = rows.dropna(subset=["ra", "dec", "inj_ra", "inj_dec"])
truth = SkyCoord(ra=rows.inj_ra.to_numpy() * u.rad, dec=rows.inj_dec.to_numpy() * u.rad)
recovered = SkyCoord(ra=rows.ra.to_numpy() * u.deg, dec=rows.dec.to_numpy() * u.deg)
rows["separation_deg"] = truth.separation(recovered).deg
print(rows[["id", "rho", "separation_deg"]])
fig, ax = plt.subplots()
ax.scatter(rows["rho"], rows["separation_deg"])
ax.set(xlabel="Coherent ranking statistic", ylabel="Sky separation [deg]")


The trigger association is not a complete list of scheduled or missed sources.
Use `pycwb simulation-summary --work-dir RUN` and `pycwb match-simulations`
with `--how right` for recovery accounting. For comparisons between different
search masks/networks use `examples/tutorials/compare.yaml`, which preserves
run identities and scheduled truth. Keep the catalog's jobs manifest with it.
